
# The Ceiling, and How to Raise It

Two questions, one after the other.

**How good can this model get?** Usually unanswerable. Here it is answerable,
because the generating process is known. Every customer has hidden traits --
an `activity_level`, a `drift` -- that determine their true order rate, and
those are dropped before the CSVs are written. A model given the true expected
order count is an **oracle**, and nothing built from observed behaviour can
beat it. Its score is the ceiling.

**If the ceiling is too low, what raises it?** Not features. The target. The
second half of this notebook shows a 30-day churn window is mostly measuring
coin flips, and that widening it buys more than any feature engineering did.


## 1. Setup

In [1]:

import sys
sys.path.insert(0, "..")

import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import brier_score_loss, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder, StandardScaler

from src.evaluation import bootstrap_metric, format_ci, paired_bootstrap
from src.features import PREDICTION_DATE, build_features
from src.generate_data import PREDICTION_DAY, START_DATE, expected_orders, generate
from src.scoring import (
    CATEGORICAL_FEATURES, MODEL_NUMERIC_FEATURES, NUMERIC_FEATURES,
    add_rate_features,
)

RANDOM_STATE = 42

def make_pipeline(numeric=None, categorical=CATEGORICAL_FEATURES, derive=True):
    steps = []
    if derive:
        steps.append(("rates", FunctionTransformer(add_rate_features)))
    transformers = [("num", Pipeline([
        ("i", SimpleImputer(strategy="median")), ("s", StandardScaler()),
    ]), list(numeric if numeric is not None else MODEL_NUMERIC_FEATURES))]
    if categorical:
        transformers.append(("cat", Pipeline([
            ("i", SimpleImputer(strategy="most_frequent")),
            ("o", OneHotEncoder(handle_unknown="ignore")),
        ]), list(categorical)))
    steps += [
        ("pre", ColumnTransformer(transformers)),
        ("model", GradientBoostingClassifier(
            random_state=RANDOM_STATE, learning_rate=0.03, max_depth=2,
            min_samples_leaf=10, n_estimators=100)),
    ]
    return Pipeline(steps)

customers = pd.read_csv("../data/customers.csv")
orders = pd.read_csv("../data/orders.csv")
events = pd.read_csv("../data/website_events.csv")
print("loaded")


loaded



## 2. The oracle

Churn is "no order in the window", so if the true expected order count in that
window is `L`, the true churn probability is `exp(-L)`. That is monotone in
`L`, so ranking by `-L` is the Bayes-optimal ranking, and its ROC-AUC is the
ceiling for any model of this target.

`expected_orders()` computes `L` exactly from the hidden traits.


In [2]:

hidden, _, _, _ = generate()

features = build_features(customers, orders, events)
y = features["churn"]

lam = expected_orders(
    hidden.set_index("customer_id").loc[features["customer_id"]].reset_index(),
    PREDICTION_DAY + 1, PREDICTION_DAY + 30,
)

print(f"prediction date     : {PREDICTION_DATE.date()}")
print(f"eligible customers  : {len(features):,}")
print(f"churn rate          : {y.mean():.4f}")
print(f"expected orders in the 30-day window: "
      f"mean {lam.mean():.3f}, median {np.median(lam):.3f}")
print()
print("A mean of well under one expected order per customer is the whole")
print("problem: the label is mostly Poisson noise.")


prediction date     : 2025-12-31
eligible customers  : 4,418
churn rate          : 0.6464
expected orders in the 30-day window: mean 0.525, median 0.418

A mean of well under one expected order per customer is the whole
problem: the label is mostly Poisson noise.


## 3. The ladder at a 30-day window

In [3]:

X = features[NUMERIC_FEATURES + CATEGORICAL_FEATURES]

idx = np.arange(len(X))
i_tr, i_tmp = train_test_split(idx, test_size=0.30, stratify=y, random_state=RANDOM_STATE)
i_val, i_te = train_test_split(i_tmp, test_size=0.50, stratify=y.iloc[i_tmp],
                               random_state=RANDOM_STATE)

y_te = y.iloc[i_te].to_numpy()

raw = make_pipeline(numeric=NUMERIC_FEATURES, derive=False).fit(X.iloc[i_tr], y.iloc[i_tr])
shipped = make_pipeline().fit(X.iloc[i_tr], y.iloc[i_tr])

results = {
    "raw counts only": bootstrap_metric(y_te, raw.predict_proba(X.iloc[i_te])[:, 1]),
    "shipped (raw + rate features)": bootstrap_metric(y_te, shipped.predict_proba(X.iloc[i_te])[:, 1]),
    "ORACLE (true expected orders)": bootstrap_metric(y_te, -lam[i_te]),
}

for name, r in results.items():
    print(f"{name:<32} {format_ci(r, 4)}")

gap = results["ORACLE (true expected orders)"]["estimate"] - results["shipped (raw + rate features)"]["estimate"]
print(f"\nremaining headroom: {gap:+.4f}")


raw counts only                  0.6556 [0.6132, 0.6992]
shipped (raw + rate features)    0.6725 [0.6290, 0.7146]
ORACLE (true expected orders)    0.7274 [0.6869, 0.7669]

remaining headroom: +0.0550



Those intervals overlap heavily, which invites the wrong conclusion. Overlap
is **not** a significance test. Both models score the same rows, so their
errors are correlated and the right comparison is paired.


In [4]:

paired = paired_bootstrap(y_te, shipped.predict_proba(X.iloc[i_te])[:, 1], -lam[i_te])

print(f"oracle - shipped : {format_ci(paired, 4)}")
print(f"oracle wins in   : {paired['win_rate']:.1%} of resamples")
print(f"significant      : {paired['significant']}")
print()
print("Overlapping intervals, and yet the difference is unambiguous.")


oracle - shipped : 0.0550 [0.0254, 0.0850]
oracle wins in   : 100.0% of resamples
significant      : True

Overlapping intervals, and yet the difference is unambiguous.



## 4. Raising the ceiling

The 30-day window is the problem. With well under one expected order per
customer, whether a given customer happens to order is close to a coin flip
even when their rate is known perfectly.

Widening the window means more expected orders per customer, so the label
carries more information about the underlying rate. Below, the same features
and the same model, evaluated against target windows of 30, 60 and 90 days.


In [5]:

rows = []

for window in (30, 60, 90):
    f = build_features(customers, orders, events, target_window_days=window)
    yw = f["churn"]
    Xw = f[NUMERIC_FEATURES + CATEGORICAL_FEATURES]

    lw = expected_orders(
        hidden.set_index("customer_id").loc[f["customer_id"]].reset_index(),
        PREDICTION_DAY + 1, PREDICTION_DAY + window,
    )

    j = np.arange(len(Xw))
    j_tr, j_tmp = train_test_split(j, test_size=0.30, stratify=yw, random_state=RANDOM_STATE)
    _, j_te = train_test_split(j_tmp, test_size=0.50, stratify=yw.iloc[j_tmp],
                               random_state=RANDOM_STATE)

    pipe = make_pipeline().fit(Xw.iloc[j_tr], yw.iloc[j_tr])
    yy = yw.iloc[j_te].to_numpy()

    m = bootstrap_metric(yy, pipe.predict_proba(Xw.iloc[j_te])[:, 1])
    o = bootstrap_metric(yy, -lw[j_te])

    rows.append({
        "window_days": window,
        "churn_rate": round(float(yw.mean()), 4),
        "mean_expected_orders": round(float(lw.mean()), 3),
        "model_auc": round(m["estimate"], 4),
        "ceiling_auc": round(o["estimate"], 4),
        "headroom": round(o["estimate"] - m["estimate"], 4),
    })

pd.DataFrame(rows).set_index("window_days")


,churn_rate,mean_expected_orders,model_auc,ceiling_auc,headroom
window_days,,,,,
30,0.6464,0.525,0.6725,0.7274,0.0550
60,0.4556,1.057,0.7006,0.7690,0.0684
90,0.3354,1.595,0.7306,0.8293,0.0987


In [6]:

first, last = rows[0], rows[-1]
print(f"ceiling, 30d -> 90d : {last['ceiling_auc'] - first['ceiling_auc']:+.4f}")
print(f"model,   30d -> 90d : {last['model_auc'] - first['model_auc']:+.4f}")
print()
print("For scale, the rate features above were worth +0.017.")


ceiling, 30d -> 90d : +0.1019
model,   30d -> 90d : +0.0581

For scale, the rate features above were worth +0.017.



## 5. What this means

**Reframing the target is worth several times what feature engineering was
worth.** Moving from a 30-day to a 90-day window raises the achievable
ceiling by about 0.10 AUC, against +0.017 for the best feature change above.
No amount of work on features could have found
that, because it was never in the features -- it was in the question.

Note the headroom *widens* as the window grows. A longer window does not just
make the problem easier; it creates signal the current feature set does not
yet exploit, so feature work becomes worthwhile again *after* the target is
fixed. Order matters.

The catch is that a 90-day window answers a different business question, and a
slower one: you wait 90 days to learn whether a prediction was right. Whether
that trade is worth making is a product decision, not a modelling one. What
this analysis provides is the price.
